# Arctic Landsat Optimised SST Viewer

Interactive notebook for loading and visualising Landsat-derived optimised sea surface
temperatures (SST) over Arctic regions using the **`landsat_sst`** module.

The algorithm applies a single-channel interaction-term regression trained on libRadtran
radiative-transfer simulations and validated against EasyCORA Arctic in-situ data.

The method is described in Ing et al., (2025, in review). Please cite the following reference 
if uising this method

**Ing, R., Nienow, P., Slater, D., Medina-Lopez, E. (in review). Investigating the use of 
Landsat-derived Sea Surface Temperatures as a proxy for changes in ocean forcing of Greenland’s 
marine-terminating outlet glaciers. Journal of Glaciology.**



| | |
|---|---|
| **Sensors** | Landsat 4, 5, 7, 8, 9 (Collection 2 Tier 1) |
| **Period** | 1982 – present |
| **Output** | SST (°C), brightness temperature (K), MERRA-2 TCWV (kg m⁻²) |
| **Water mask** | QA_PIXEL bits 0–5, 7–15 (cloud, shadow, snow, fill, cirrus for L8/L9) |

## Dependencies

```bash
pip install earthengine-api geemap folium pandas
```

Earth Engine authentication (first time only):
```bash
earthengine authenticate
```

In [ ]:
import ee
import geemap
import folium
import pandas as pd
from datetime import datetime, timezone

# --- Earth Engine authentication ---
# First time: uncomment and run the line below, then re-run this cell.
# ee.Authenticate()

ee.Initialize()

print('Earth Engine initialised.')
print(f'earthengine-api version: {ee.__version__}')
print(f'geemap version         : {geemap.__version__}')

In [ ]:
import sys
from pathlib import Path

# Ensure the repo root is on the Python path so the module is importable
# from anywhere (adjust the path below if needed).
REPO_ROOT = Path('.').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from landsat_sst import get_collection, build_water_mask, RGB_BANDS
from landsat_sst.collection import COLLECTION_INFO

print('landsat_sst module loaded successfully.')
print('  get_collection   — build SST ImageCollection per sensor')
print('  build_water_mask — QA_PIXEL water + quality mask')
print('  RGB_BANDS        — true-colour band names per sensor')
print('  COLLECTION_INFO  — asset IDs and band lists per sensor')

## 1.  Study Area

Adjust `GLACIER_NAME`, `MAP_CENTER`, and `roi` to your site of interest.

In [ ]:
# ── Study area configuration ──────────────────────────────────────────────
GLACIER_NAME = 'Store Glacier'
MAP_CENTER   = [70.55, -51.3]          # [lat, lon] — used to centre folium maps

roi = ee.Geometry.Point(MAP_CENTER[::-1]).buffer(5000)  # 5 km buffer around the point

# All five Landsat sensors are loaded and merged into one SST ImageCollection.
ALL_SENSORS = ['L4', 'L5', 'L7', 'L8', 'L9']

# Date window for archive browsing (summer melt season)
BROWSE_START = '2022-06-01'
BROWSE_END   = '2022-09-30'

print(f'Site   : {GLACIER_NAME}')
print(f'Sensors: {ALL_SENSORS}')
print(f'Window : {BROWSE_START} – {BROWSE_END}')

## 2.  Browse the Landsat Archive

List available scenes from all five sensors over the ROI for the chosen date window,
sorted by cloud cover so the clearest scenes appear first.

In [ ]:
def _tag_sensor(sensor_key):
    return lambda img: img.set('sensor_key', sensor_key)

archive = None
for s in ALL_SENSORS:
    a = (
        ee.ImageCollection(COLLECTION_INFO[s]['SR'])
          .filterBounds(roi)
          .filterDate(BROWSE_START, BROWSE_END)
          .filter(ee.Filter.lt('CLOUD_COVER', 30))
          .map(_tag_sensor(s))
    )
    archive = a if archive is None else archive.merge(a)

result = archive.reduceColumns(
    reducer=ee.Reducer.toList().repeat(4),
    selectors=['system:time_start', 'CLOUD_COVER', 'system:index', 'sensor_key']
).getInfo()

if result and result.get('list') and result['list'][0]:
    timestamps, clouds, ids, sensors = result['list']
    scenes_df = pd.DataFrame({
        'date'     : [datetime.fromtimestamp(t / 1000, tz=timezone.utc).strftime('%Y-%m-%d')
                      for t in timestamps],
        'sensor'   : sensors,
        'cloud_%'  : [round(c, 1) for c in clouds],
        'scene_id' : ids,
    }).sort_values('cloud_%').reset_index(drop=True)
    print(f'Found {len(scenes_df)} scenes (all sensors, {BROWSE_START}–{BROWSE_END}, cloud < 30%)')
    print(scenes_df.head(12).to_string(index=False))
else:
    print('No scenes found.  Try expanding BROWSE_START/BROWSE_END or relaxing cloud cover.')
    scenes_df = pd.DataFrame()

## 3.  Load Optimised SST

Builds per-sensor SST collections (MERRA-2 TCWV lookup + RobustScaler normalisation
+ interaction-term regression + per-sensor bias correction D) and merges them into
one `ee.ImageCollection`.  
Change `TARGET_DATE` to any date shown above to load a different day.

In [ ]:
# ── Pick a scene ───────────────────────────────────────────────────────────────────────────
# Auto-select the clearest scene from the browse results,
# or set TARGET_DATE manually: e.g. TARGET_DATE = '2022-08-01'
if not scenes_df.empty:
    TARGET_DATE = scenes_df.iloc[0]['date']
else:
    TARGET_DATE = '2022-08-01'   # fallback

print(f'Loading multi-sensor SST for {TARGET_DATE} ...')

# One-day window: loads all sensors' scenes for the target date
scene_end = ee.Date(TARGET_DATE).advance(1, 'day').format('YYYY-MM-dd').getInfo()

# Build per-sensor SST collections, tag each image with its sensor key,
# then merge all into a single ImageCollection.
def _build_sensor_col(sensor, start, end, geom):
    return (
        get_collection(sensor, start, end, geom)
          .map(lambda img: img.set('sensor_key', sensor))
    )

col_parts = [_build_sensor_col(s, TARGET_DATE, scene_end, roi) for s in ALL_SENSORS]
col = col_parts[0]
for c in col_parts[1:]:
    col = col.merge(c)

count = col.size().getInfo()

if count == 0:
    print('No SST scenes returned.  Change TARGET_DATE.')
else:
    image    = col.first()
    date_str = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    SENSOR   = image.get('sensor_key').getInfo()
    bands    = image.bandNames().getInfo()

    # Sanity-check MERRA-2 TCWV (sentinel -999 means no match was found)
    tcwv_min = (
        image.select('TCWV')
             .reduceRegion(ee.Reducer.min(), roi, scale=50000)
             .getInfo().get('TCWV', -999)
    )
    tcwv_ok = (tcwv_min is not None) and (float(tcwv_min) > -998)

    print(f'Scenes in collection : {count}  (across all sensors on {TARGET_DATE})')
    print(f'First scene date     : {date_str}')
    print(f'First scene sensor   : Landsat {SENSOR[1:]}')
    print(f'Bands                : {bands}')
    print(f'MERRA-2 TCWV         : {"OK  (min = " + str(round(float(tcwv_min), 1)) + " kg m-2)" if tcwv_ok else "WARNING — sentinel value detected; SST unreliable"}')

In [ ]:
# ── Build visualisation layers ──────────────────────────────────────────────
# Comprehensive water mask (matches Python training pipeline)
qa        = image.select('QA_PIXEL')
water     = build_water_mask(qa, SENSOR)

sst_masked = image.select('SST').updateMask(water)
bt_masked  = image.select('bt_K').updateMask(water)
rgb        = image.select(RGB_BANDS[SENSOR])

# Visualisation parameters
rgb_vis = {'min': 0.0, 'max': 0.3, 'gamma': 1.4}

# SST: blue (cold) → yellow (neutral) → red (warm)
sst_vis = {
    'min': -2, 'max': 12,
    'palette': [
        '#313695', '#4575b4', '#74add1', '#abd9e9', '#e0f3f8',
        '#ffffbf',
        '#fee090', '#fdae61', '#f46d43', '#d73027', '#a50026'
    ]
}

# Brightness temperature (for algorithm QC)
bt_vis = {
    'min': 271, 'max': 285,
    'palette': ['#081d58', '#253494', '#225ea8', '#1d91c0', '#41b6c4',
                '#7fcdbb', '#c7e9b4', '#edf8b1', '#ffffd9']
}

LAYER_TITLE = f'Landsat {SENSOR[1:]} — {date_str}'
print(f'Layers ready for: {LAYER_TITLE}')
print(f'  True colour  : {RGB_BANDS[SENSOR]}')
print(f'  SST display  : {sst_vis["min"]} to {sst_vis["max"]} °C')

## 4.  Visualise with geemap

The cells below display the true-colour composite and the optimised SST on
interactive geemap maps.  Use the layer control panel (top-right) to toggle
layers.  

In [ ]:
# SST with colourbar
Map_sst = geemap.Map()
Map_sst.centerObject(roi, 9)
Map_sst.addLayer(rgb.clip(roi),        rgb_vis, 'True Colour',        False)
Map_sst.addLayer(bt_masked,  bt_vis,  'Brightness Temp (K)', False)
Map_sst.addLayer(sst_masked, sst_vis, f'Optimised SST (°C) — {LAYER_TITLE}')
Map_sst.add_colorbar(
    sst_vis,
    label='Sea Surface Temperature (°C)',
    layer_name=f'SST (°C) — {LAYER_TITLE}'
)
Map_sst

In [ ]:
# Side-by-side split map: true colour (left) vs optimised SST (right)
left_layer  = geemap.ee_tile_layer(rgb,        rgb_vis, 'True Colour')
right_layer = geemap.ee_tile_layer(sst_masked, sst_vis, 'Optimised SST (°C)')

Map_split = geemap.Map()
Map_split.centerObject(roi, 9)
Map_split.split_map(left_layer=left_layer, right_layer=right_layer)
Map_split

## 5.  Folium Alternative

If you prefer a lightweight map without the full geemap dependency, the cell
below builds the same visualisation using **folium** with GEE tile URLs.
Toggle layers using the layer control (top-right of the map).

In [ ]:
# Request tile URLs from Earth Engine (requires EE initialisation above)
rgb_tile = rgb.clip(roi).getMapId(rgb_vis)
sst_tile = sst_masked.getMapId(sst_vis)
bt_tile  = bt_masked.getMapId(bt_vis)

m = folium.Map(location=MAP_CENTER, zoom_start=9, tiles='CartoDB positron')

folium.TileLayer(
    tiles=rgb_tile['tile_fetcher'].url_format,
    attr='USGS Landsat / Google Earth Engine',
    name=f'True Colour — {LAYER_TITLE}',
    overlay=True, show=True
).add_to(m)

folium.TileLayer(
    tiles=sst_tile['tile_fetcher'].url_format,
    attr='Optimised SST — Ryan Ing / University of Edinburgh',
    name=f'Optimised SST (°C) — {LAYER_TITLE}',
    overlay=True, show=True
).add_to(m)

folium.TileLayer(
    tiles=bt_tile['tile_fetcher'].url_format,
    attr='Brightness Temperature — USGS Landsat / Google Earth Engine',
    name=f'Brightness Temp (K) — {LAYER_TITLE}',
    overlay=True, show=False
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m

## 6.  Statistics Summary

Compute summary statistics over the ROI for the quality-masked water pixels.

In [ ]:
# Reduce SST and BT over the ROI
stats = (
    sst_masked
      .addBands(bt_masked)
      .reduceRegion(
          reducer=(
              ee.Reducer.mean()
                .combine(ee.Reducer.stdDev(),               sharedInputs=True)
                .combine(ee.Reducer.percentile([5, 50, 95]), sharedInputs=True)
                .combine(ee.Reducer.count(),                sharedInputs=True)
          ),
          geometry=roi,
          scale=30,
          maxPixels=1e9,
          bestEffort=True,
      )
      .getInfo()
)

n = int(stats.get('SST_count', 0))
print(f'SST Statistics  |  {GLACIER_NAME}  |  Landsat {SENSOR[1:]}  |  {date_str}')
print(f'  Valid water pixels : {n:,}')
if n > 0:
    print(f'  Mean              : {stats["SST_mean"]:.2f} °C')
    print(f'  Std Dev           : {stats["SST_stdDev"]:.2f} °C')
    print(f'  5th percentile    : {stats["SST_p5"]:.2f} °C')
    print(f'  Median            : {stats["SST_p50"]:.2f} °C')
    print(f'  95th percentile   : {stats["SST_p95"]:.2f} °C')
    print(f'  BT mean           : {stats["bt_K_mean"]:.2f} K')
else:
    print('  No valid water pixels in the ROI (check date or cloud cover).')

## 7.  Multi-Sensor Collection

All five Landsat sensors are loaded and merged automatically.  The correct algorithm
coefficients, scaler parameters, and sensor-specific bias correction (D) are applied
per sensor inside `get_collection`.  The `SENSOR` variable is detected from the first
image found and used for display; the full merged `col` contains scenes from whichever
sensors had coverage on `TARGET_DATE`.

| Sensor | Active years | Thermal band | Bias correction D |
|--------|-------------|--------------|-------------------|
| `L4`   | 1982 – 1993 | B6 (TM) | 1.049 °C (borrowed from L5) |
| `L5`   | 1984 – 2013 | B6 (TM) | 1.049 °C |
| `L7`   | 1999 – 2022 | B6_VCID_1 (ETM+) | 0.769 °C |
| `L8`   | 2013 – present | B10 (TIRS) | 1.328 °C |
| `L9`   | 2021 – present | B10 (TIRS-2) | 1.152 °C |

### Algorithm formula

$$
\text{SST}_K = A \cdot \tilde{T}_b + B \cdot (\tilde{T}_b \cdot \widetilde{\text{TCWV}}) + C + D
$$

where $\tilde{T}_b = (T_b - \text{median}_{T_b}) / \text{IQR}_{T_b}$ and
$\widetilde{\text{TCWV}} = (\text{TCWV} - 10.62) / 6.40$ (RobustScaler normalisation).